# Diabetic Retinopathy Stage Detection
**BSc Computer Science — Computer Vision Module Coursework**

This notebook implements a complete pipeline for detecting and staging Diabetic Retinopathy (DR) using
deep learning (EfficientNetB3 transfer learning) with explainability (Grad-CAM), embedding-based
similar-case retrieval, and a multi-agent clinical decision pipeline.

**Dataset:** Combined DR Dataset (APTOS + IDRiD + Messidor-2 + EyePACS subset)
**Classes (ICDR 0–4 scale):**
- 0: No DR
- 1: Mild NPDR
- 2: Moderate NPDR
- 3: Severe NPDR
- 4: Proliferative DR (PDR)

---
## Pipeline Sections
1. Setup & Data Acquisition
2. Preprocessing
3. Data Augmentation & Class Balancing
4. Train/Validation/Test Split
5. Model — CNN with Transfer Learning (EfficientNetB3)
6. Training Strategy
7. Evaluation
8. Explainability — Grad-CAM
9. Innovation A — Embedding-Based Similar-Case Retrieval
10. Innovation B — Multi-Agent Clinical Decision Pipeline
11. UI — Gradio Interface
---

## Section 1: Setup & Data Acquisition
**Goal:** Install dependencies, configure all project constants in one place (Config class),
verify GPU, download the dataset via the Kaggle API, and load/validate the labels —
including a robust fallback to folder-based labelling if no CSV is found,
corrupt-image filtering, and a class-distribution summary.

In [ ]:
# ============================================================
# SECTION 1.1 — INSTALL DEPENDENCIES
# Run this cell once. Restart the runtime if prompted after install.
# ============================================================
!pip install -q kaggle scikit-learn seaborn pillow opencv-python-headless gradio matplotlib tqdm pandas numpy tensorflow

In [ ]:
# ============================================================
# SECTION 1.2 — GLOBAL CONFIGURATION (single source of truth)
# All magic numbers and tunable hyperparameters live here.
# Modify Config values rather than hunting through code.
# ============================================================

import os
import random
import numpy as np
import tensorflow as tf


class Config:
    """
    Central configuration object.

    All project-wide constants are defined here so they can be changed in
    one place. Keeping constants here also makes experiments reproducible:
    bump SEED and everything downstream uses the new value automatically.
    """

    # --- Reproducibility ---
    SEED: int = 42

    # --- Dataset paths ---
    KAGGLE_DATASET: str = "harsha1289/combined-dr-dataset-aptosidridmessidoreyepacs"
    DATA_DIR: str       = "/content/dr_data"
    TRAIN_DIR: str      = "/content/dr_data/train"
    # CSV filenames to try, in priority order
    LABEL_CANDIDATES: list = ["train.csv", "labels.csv", "trainLabels.csv"]

    # --- Image preprocessing ---
    IMG_SIZE: int          = 224   # EfficientNetB3 default input resolution
    BEN_GRAHAM_SIGMA: int  = 10    # Gaussian blur radius for Ben Graham enhancement
    BEN_GRAHAM_ALPHA: float = 4.0  # weight on original image
    BEN_GRAHAM_BETA: float  = -4.0 # weight on blurred image (subtracted)
    BEN_GRAHAM_GAMMA: float = 128  # additive bias to centre pixel distribution

    # --- Class labels (ICDR 0-4 scale) ---
    CLASS_NAMES: list = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
    NUM_CLASSES: int  = 5

    # --- Train / Val / Test split ---
    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float   = 0.15
    TEST_RATIO: float  = 0.15   # three ratios must sum to 1.0

    # --- Model / Training hyperparameters ---
    BATCH_SIZE: int      = 32
    PHASE1_EPOCHS: int   = 15    # Phase 1: frozen base, train head only
    PHASE2_EPOCHS: int   = 25    # Phase 2: fine-tune top N base layers
    PHASE1_LR: float     = 1e-3  # higher LR is safe when base is frozen
    PHASE2_LR: float     = 1e-5  # very low LR prevents catastrophic forgetting
    DROPOUT_RATE: float  = 0.3   # applied before final softmax
    DENSE_UNITS: int     = 256   # units in intermediate dense layer
    UNFREEZE_TOP_N: int  = 30    # base-model layers to unfreeze in phase 2

    # --- Callbacks ---
    ES_PATIENCE: int     = 5     # early stopping patience (val_loss)
    RLROP_FACTOR: float  = 0.5   # LR reduction factor on plateau
    RLROP_PATIENCE: int  = 3     # patience before reducing LR

    # --- GovernanceAgent threshold ---
    CONFIDENCE_THRESHOLD: float = 0.70  # flag predictions below this confidence

    # --- Output paths ---
    CHECKPOINT_DIR: str  = "/content/checkpoints"
    REPORTS_DIR: str     = "/content/report_images"
    EMBEDDINGS_PATH: str = "/content/embeddings.npz"


def set_all_seeds(seed: int = Config.SEED) -> None:
    """
    Set random seeds for Python, NumPy, and TensorFlow.

    Seeds are set globally so all downstream random operations — data
    shuffling, weight initialisation, augmentation — produce the same
    results across runs, making experiments reproducible.

    Args:
        seed: Integer seed value. Defaults to Config.SEED.
    """
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[Config] All random seeds set to {seed}.")


set_all_seeds()

In [ ]:
# ============================================================
# SECTION 1.3 — IMPORT ALL LIBRARIES
# Centralised imports make missing-dependency errors easy to spot
# and prevent the notebook from failing midway through a long run.
# ============================================================

import sys
import glob
import shutil
import warnings
import pathlib
from typing import List, Tuple, Optional, Dict

import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")  # suppress non-critical deprecation warnings

# Create all output directories upfront — avoids failures in later sections
for _dir in [Config.CHECKPOINT_DIR, Config.REPORTS_DIR, Config.DATA_DIR]:
    os.makedirs(_dir, exist_ok=True)

print(f"Python     : {sys.version}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print("All libraries imported successfully.")

In [ ]:
# ============================================================
# SECTION 1.4 — GPU VERIFICATION
# Training on CPU for a 21k-image dataset takes many hours.
# We detect GPUs here and warn loudly if none is found.
# In Colab: Runtime > Change runtime type > GPU (T4 or A100).
# ============================================================

def verify_gpu() -> None:
    """
    Detect and report available GPU devices.

    Enables memory growth on each GPU to prevent TensorFlow from
    grabbing all VRAM at startup, which can cause OOM errors when
    sharing a Colab GPU with other processes.

    Prints a warning (but does not raise) if no GPU is found, so
    the notebook can still be used for small-batch testing on CPU.
    """
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            # Incremental VRAM allocation rather than reserving all memory up-front
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[GPU] {len(gpus)} GPU(s) detected:")
        for g in gpus:
            print(f"      {g.name}")
        os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
    else:
        print(
            "[WARNING] No GPU detected. Training will be very slow on CPU.\n"
            "          In Google Colab: Runtime > Change runtime type > GPU."
        )


verify_gpu()

In [ ]:
# ============================================================
# SECTION 1.5 — KAGGLE API DOWNLOAD
# The Kaggle API requires a personal API token (kaggle.json).
# In Colab this cell prompts an upload dialog.
# Locally it reads kaggle.json from the current working directory.
# ============================================================

def setup_kaggle_credentials() -> None:
    """
    Place kaggle.json in ~/.kaggle/kaggle.json with correct permissions.

    The Kaggle CLI refuses to run if permissions on kaggle.json are too
    permissive (it is a security token). We enforce 600 (owner read/write).

    Raises:
        FileNotFoundError: If kaggle.json cannot be found or uploaded.
    """
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    target = os.path.join(kaggle_dir, "kaggle.json")

    if os.path.exists(target):
        print("[Kaggle] kaggle.json already in place — skipping upload.")
        return

    try:
        from google.colab import files  # type: ignore
        print("[Kaggle] Please upload your kaggle.json when the dialog appears...")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError(
                "kaggle.json was not found in the uploaded files. "
                "Download it from https://www.kaggle.com/settings > API > Create New Token."
            )
        shutil.move("kaggle.json", target)
    except ImportError:
        # Not in Colab — look for kaggle.json in the current directory
        if os.path.exists("kaggle.json"):
            shutil.copy("kaggle.json", target)
        else:
            raise FileNotFoundError(
                "kaggle.json not found in the current directory. "
                "Place it here or run in Colab for an upload prompt."
            )

    # Restrict permissions — required by the Kaggle CLI security check
    os.chmod(target, 0o600)
    print(f"[Kaggle] Credentials saved to {target} with 600 permissions.")


def download_dataset(force_redownload: bool = False) -> None:
    """
    Download and extract the Combined DR Dataset from Kaggle.

    Idempotent: skips the download if images are already present on disk.
    Set force_redownload=True to re-fetch the dataset regardless.

    Args:
        force_redownload: Re-download even if data already exists on disk.

    Raises:
        RuntimeError: If the Kaggle CLI returns a non-zero exit code.
    """
    existing = (
        glob.glob(os.path.join(Config.DATA_DIR, "**", "*.jpeg"), recursive=True)
        + glob.glob(os.path.join(Config.DATA_DIR, "**", "*.jpg"),  recursive=True)
        + glob.glob(os.path.join(Config.DATA_DIR, "**", "*.png"),  recursive=True)
    )
    if existing and not force_redownload:
        print(
            f"[Dataset] {len(existing):,} images already in {Config.DATA_DIR}. "
            "Skipping download. Pass force_redownload=True to override."
        )
        return

    print(f"[Dataset] Downloading '{Config.KAGGLE_DATASET}' — this may take several minutes...")
    # --unzip extracts automatically; -q suppresses verbose progress output
    exit_code = os.system(
        f"kaggle datasets download -d {Config.KAGGLE_DATASET} "
        f"-p {Config.DATA_DIR} --unzip -q"
    )
    if exit_code != 0:
        raise RuntimeError(
            f"Kaggle download failed (exit code {exit_code}). "
            "Check your kaggle.json credentials and the dataset slug."
        )
    print(f"[Dataset] Download complete -> {Config.DATA_DIR}")


setup_kaggle_credentials()
download_dataset()

In [ ]:
# ============================================================
# SECTION 1.6 — DISCOVER DATASET STRUCTURE
# Print the top-level directory tree so we know exactly what Kaggle
# delivered before attempting label loading.
# ============================================================

def print_directory_tree(root: str, max_depth: int = 3, max_items: int = 20) -> None:
    """
    Print a human-readable directory tree for quick structure inspection.

    Args:
        root:      Root directory to start from.
        max_depth: Maximum folder depth to recurse into.
        max_items: Max items shown per directory (prevents flooding output).
    """
    root_path = pathlib.Path(root)
    if not root_path.exists():
        print(f"[Tree] Path does not exist: {root}")
        return

    def _recurse(path: pathlib.Path, depth: int, prefix: str) -> None:
        if depth > max_depth:
            return
        try:
            children = sorted(path.iterdir())
        except PermissionError:
            return
        dirs  = [c for c in children if c.is_dir()]
        files = [c for c in children if c.is_file()]
        items = dirs + files
        truncated = len(items) > max_items
        items = items[:max_items]
        for i, item in enumerate(items):
            is_last = (i == len(items) - 1) and not truncated
            connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
            suffix = "/" if item.is_dir() else ""
            print(f"{prefix}{connector}{item.name}{suffix}")
            if item.is_dir():
                ext = "    " if is_last else "\u2502   "
                _recurse(item, depth + 1, prefix + ext)
        if truncated:
            print(f"{prefix}    ... (showing first {max_items} items)")

    print(f"\n[Tree] {root}/")
    _recurse(root_path, 1, "")


print_directory_tree(Config.DATA_DIR)

In [ ]:
# ============================================================
# SECTION 1.7 — LABEL LOADING (CSV-first, folder fallback)
#
# Loading strategy:
#   1. Search for a recognised CSV file (Config.LABEL_CANDIDATES).
#   2. If found: parse and normalise all label values to ICDR 0-4.
#   3. If not found: derive labels from subfolder names.
#
# Messidor-2 note:
#   Messidor-2 originally used a 4-level scale (grades 0-3) where
#   grade 3 covers both Severe and Proliferative DR. The Kaggle
#   combined dataset re-maps everything to ICDR 0-4 before upload, so
#   inconsistencies should be rare — but we validate and flag any that
#   slip through rather than silently accepting them.
# ============================================================

# Column name candidates across sub-datasets
_LABEL_COL_CANDIDATES = ["diagnosis", "label", "level", "dr_grade", "grade", "class"]
_IMG_COL_CANDIDATES   = ["id_code", "image", "filename", "image_name", "id"]

# Folder name -> ICDR integer mapping (covers common naming conventions)
_FOLDER_LABEL_MAP = {
    "0": 0, "no_dr": 0, "nodr": 0, "normal": 0,
    "1": 1, "mild": 1,
    "2": 2, "moderate": 2,
    "3": 3, "severe": 3,
    "4": 4, "proliferative": 4, "pdr": 4,
}


def _find_csv(search_root: str) -> Optional[str]:
    """
    Search for a label CSV file inside search_root.

    Tries Config.LABEL_CANDIDATES at the root level first, then falls
    back to a recursive glob one level deeper.

    Args:
        search_root: Directory to search.

    Returns:
        Absolute path to the first matching CSV, or None if not found.
    """
    for candidate in Config.LABEL_CANDIDATES:
        fp = os.path.join(search_root, candidate)
        if os.path.exists(fp):
            return fp
    for candidate in Config.LABEL_CANDIDATES:
        matches = glob.glob(os.path.join(search_root, "**", candidate), recursive=True)
        if matches:
            return matches[0]
    return None


def _normalise_csv_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect and standardise label and image-path columns in a raw CSV DataFrame.

    Handles differing column names (APTOS uses 'diagnosis', EyePACS uses 'level',
    IDRiD uses 'DR_grade') and string labels ("Mild", "moderate") across
    sub-datasets. Maps all values to ICDR integers 0-4.

    Args:
        df: Raw DataFrame loaded from pd.read_csv().

    Returns:
        DataFrame with exactly two columns: 'filepath' (str) and 'label' (int 0-4).

    Raises:
        ValueError: If no recognisable label or image-filename column is found.
    """
    df.columns = [c.strip().lower() for c in df.columns]

    label_col = next((c for c in _LABEL_COL_CANDIDATES if c in df.columns), None)
    if label_col is None:
        raise ValueError(
            f"No label column found. Columns present: {list(df.columns)}. "
            f"Expected one of: {_LABEL_COL_CANDIDATES}"
        )

    img_col = next((c for c in _IMG_COL_CANDIDATES if c in df.columns), None)
    if img_col is None:
        raise ValueError(
            f"No image-filename column found. Columns present: {list(df.columns)}. "
            f"Expected one of: {_IMG_COL_CANDIDATES}"
        )

    # Map string labels to ICDR integers
    string_map = {
        "no dr": 0, "no_dr": 0, "nodr": 0, "normal": 0, "0": 0,
        "mild": 1, "mild npdr": 1, "1": 1,
        "moderate": 2, "moderate npdr": 2, "2": 2,
        "severe": 3, "severe npdr": 3, "3": 3,
        "proliferative dr": 4, "pdr": 4, "proliferative": 4, "4": 4,
    }
    raw = df[label_col].astype(str).str.strip().str.lower()
    df["label"] = raw.map(string_map)

    # For any still-NaN labels, try direct numeric conversion
    nan_mask = df["label"].isna()
    if nan_mask.any():
        df.loc[nan_mask, "label"] = pd.to_numeric(
            df.loc[nan_mask, label_col], errors="coerce"
        )

    df["filepath"] = df[img_col].astype(str)
    return df[["filepath", "label"]]


def _load_from_folders(image_root: str) -> pd.DataFrame:
    """
    Build a label DataFrame from a folder-structured dataset.

    Expected structure::

        image_root/
            0/  (or No_DR/)
                image001.jpeg
            1/  (or Mild/)
                ...

    Args:
        image_root: Root directory with one subdirectory per class.

    Returns:
        DataFrame with 'filepath' (absolute) and 'label' (int 0-4).

    Raises:
        FileNotFoundError: If image_root does not exist or contains no images.
    """
    if not os.path.isdir(image_root):
        raise FileNotFoundError(f"Image root directory not found: {image_root}")

    records = []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

    for folder_name in sorted(os.listdir(image_root)):
        folder_path = os.path.join(image_root, folder_name)
        if not os.path.isdir(folder_path):
            continue
        key = folder_name.strip().lower().replace(" ", "_")
        label_int = _FOLDER_LABEL_MAP.get(key)
        if label_int is None:
            print(
                f"[Label] WARNING: folder '{folder_name}' cannot be mapped to an ICDR class "
                "and will be skipped. Add it to _FOLDER_LABEL_MAP if needed."
            )
            continue
        for fname in os.listdir(folder_path):
            if pathlib.Path(fname).suffix.lower() in valid_exts:
                records.append({
                    "filepath": os.path.join(folder_path, fname),
                    "label": label_int,
                })

    if not records:
        raise FileNotFoundError(
            f"No images found under {image_root} via folder-based labelling. "
            "Ensure subdirectory names match ICDR class names (0-4, mild, etc.)."
        )

    print(f"[Label] Folder-based labelling: {len(records):,} images found.")
    return pd.DataFrame(records)


def load_labels(data_dir: str, image_root: str) -> pd.DataFrame:
    """
    Load dataset labels using a CSV-first, folder-fallback strategy.

    After loading, validates that all labels are in {0,1,2,3,4}, drops
    rows with unmappable values, resolves relative paths to absolute paths,
    and filters out rows where the image file does not exist on disk.

    Args:
        data_dir:   Root dataset directory (searched for CSVs).
        image_root: Directory containing the actual image files.

    Returns:
        Cleaned DataFrame with columns 'filepath' (str) and 'label' (int 0-4).
    """
    csv_path = _find_csv(data_dir)

    if csv_path:
        print(f"[Label] CSV found: {csv_path}")
        raw_df = pd.read_csv(csv_path)
        print(f"[Label] {len(raw_df):,} rows loaded from CSV.")
        df = _normalise_csv_labels(raw_df)

        # Resolve relative CSV paths to absolute filesystem paths
        def _resolve(fp: str) -> str:
            if os.path.isabs(fp) and os.path.exists(fp):
                return fp
            candidate = os.path.join(image_root, fp)
            if os.path.exists(candidate):
                return candidate
            # Try adding common image extensions (some CSVs omit the extension)
            base = os.path.splitext(fp)[0]
            for ext in [".jpeg", ".jpg", ".png"]:
                c = os.path.join(image_root, base + ext)
                if os.path.exists(c):
                    return c
            return fp  # will be caught by missing-file filter below

        df["filepath"] = df["filepath"].apply(_resolve)
    else:
        print(f"[Label] No CSV found in {data_dir}. Falling back to folder-based labelling.")
        df = _load_from_folders(image_root)

    # Validate label values
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    bad_mask = ~df["label"].isin({0, 1, 2, 3, 4}) | df["label"].isna()
    if bad_mask.any():
        bad_vals = df.loc[bad_mask, "label"].unique().tolist()[:10]
        print(
            f"[Label] WARNING: {bad_mask.sum()} rows with out-of-range labels will be DROPPED. "
            f"Unmappable values: {bad_vals}\n"
            "        This may indicate Messidor-2 grading inconsistencies or annotation errors."
        )
        df = df[~bad_mask].copy()

    df["label"] = df["label"].astype(int)

    # Filter rows where the image file does not exist on disk
    exists_mask = df["filepath"].apply(os.path.exists)
    n_missing = (~exists_mask).sum()
    if n_missing > 0:
        print(f"[Label] WARNING: {n_missing} image(s) referenced in labels not found on disk. Removing.")
    df = df[exists_mask].reset_index(drop=True)
    print(f"[Label] Final usable dataset size: {len(df):,} images.")
    return df


df_labels = load_labels(Config.DATA_DIR, Config.TRAIN_DIR)
df_labels.head()